# Importing Libraries

In [ ]:
import pandas as pd

: 

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Step 1: Importing Libraries

# Step 2: Loading and Exploring the Data
# Load the dataset
data = pd.read_csv('template.csv')  # Ensure template.csv has columns: input_text, target_text
print(data.head())

# Step 3: Data Cleaning and Preprocessing
# Check for missing values
print(data.isnull().sum())
data = data.dropna()

# Step 4: Tokenization
# Use a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Tokenize the input and target texts
def tokenize_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True)
    labels = tokenizer(examples["target_text"], max_length=512, truncation=True).input_ids
    model_inputs["labels"] = labels
    return model_inputs

# Step 5: Splitting the Data
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Step 6: Model Initialization
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Step 7: Training the Model
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

trainer.train()

# Step 8: Testing the Model
def generate_description(input_json):
    input_text = f"Pet owner:{input_json['Pet owner']} Pet:{input_json['Pet']} Request Type:{input_json['Request Type']} Note:{input_json['Note']}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example Input
example_input = {
    "Pet owner": "Argie",
    "Pet": "Kenneth",
    "Request Type": "Photo",
    "Note": "I wanna see a photos of Kenneth because I really miss him"
}

# Generate Description
print(generate_description(example_input))